# Does the text of an agent skill predict its replication on GitHub? — guided notebook

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/podorize/skillreuse/blob/main/notebooks/reproduce.ipynb)

**What this notebook is.** A companion to read *next to the paper*. Each part follows one section, says what to read
first, recomputes the claim from the repository's files and logs the comparison in a **scorecard** at the end. It
needs neither the raw GitSkills data nor the GitHub API: the dates of all copies were collected once and are
published in `data/derived/collection/`.

**The study in one paragraph.** We sampled 150,000 skills from GitSkills, dated every copy outside repositories that
collect skills in bulk, and found when each skill first appeared (T0). A skill is **copied** if another
developer copied it within W days after T0. We ask whether the text of a skill predicts that (RQ1) and which characteristics carry
the signal (RQ2). Skills are split **by author**; seven algorithms are spot-checked, three are tuned, and the final
models are evaluated **once** on a holdout of authors never seen in training.

**Paper:** [PDF](https://github.com/podorize/skillreuse/blob/main/paper/latex/main.pdf) ·
[3 Methods](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md) · [4 Results](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md) · [5 Discussion](https://github.com/podorize/skillreuse/blob/main/paper/draft/05_discussion.md) ·
[Appendix A](https://github.com/podorize/skillreuse/blob/main/paper/draft/07_appendix_reproducibility.md)

| Part | Paper | What you recompute | Time |
|---|---|---|---|
| 0 | App. A | environment | ~2 min |
| 1 | §3.1–3.2, §4.1 | sample, dating of copies, T0 and author | seconds |
| 2 | §3.2, §3.4 | window rule, eligibility, target, split | seconds |
| 3 | §3.5, §4.1 | exploratory rules | seconds |
| 4 | §3.6, §4.2 | ⏱ spot-check of 7 algorithms (1 repetition) and model selection | ~1 min |
| 5 | §3.7, §4.3 | ⏱ frozen models re-applied to the holdout, control | ~1 min |
| 6 | §4.4 | robustness analyses | seconds |
| 7 | §3.8, §4.5 | interpretation | seconds |
| 8 | §5–6 | claims and scorecard | seconds |

> Cells that load the frozen models run through `!uv run skillreuse ...` in the project's **locked** environment
> (`uv.lock`), so scikit-learn versions match the saved models.

## Part 0 — Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

if not Path("src/skillreuse").exists():
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    else:
        if not Path("skillreuse").exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/podorize/skillreuse.git"], check=True)
        os.chdir("skillreuse")
print("working directory:", Path.cwd())
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

In [ ]:
!uv sync -q
!uv run skillreuse check

In [ ]:
import importlib.util, json
import numpy as np, pandas as pd
from IPython.display import Image, display

missing = [m for m in ("duckdb", "pyarrow", "sklearn") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow", "scikit-learn"], check=True)
sys.path.insert(0, str(Path("src").resolve()))
from skillreuse import study as S          # the project's own code
from skillreuse.stats import nb_test
get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 40)
R, C = Path("results"), Path("data/derived/collection")
SCORE = []

def check(part, what, paper, got, tol=0.0005, fmt="{:,.3f}"):
    ok = abs(float(paper) - float(got)) <= tol
    SCORE.append({"part": part, "check": what, "paper": fmt.format(paper), "recomputed": fmt.format(got), "ok": ok})
    print(("OK  " if ok else "DIFF"), what, "| paper", fmt.format(paper), "| recomputed", fmt.format(got))

def j(path):
    return json.loads((R / path).read_text())

def show(name, width=900):
    display(Image(filename=str(R / "figures" / name), width=width))

---
## Part 1 — Sample, dating of copies, T0 and author
> 📖 Read [§3.1–3.2 and §4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

The sample has every occurrence of 150,000 skills. For each copy outside aggregators (repositories with ≥ 100
skills), the collection recorded the first commit with exactly the skill's content and the creation of the
repository; the **arrival** is the later of the two. T0 is the earliest arrival, the author owns the origin copy.

In [ ]:
occ = pd.read_parquet(C / "sample_occurrences.parquet")
dates = pd.read_parquet(C / "dates.parquet")
print("skills:", occ.content_hash.nunique(), "| occurrences:", len(occ), "| outside aggregators:", (~occ.aggregator).sum())
d = dates.merge(occ[["content_hash", "repo_full_name", "path", "aggregator", "owner"]],
                on=["content_hash", "repo_full_name", "path"])
main = d[~d.aggregator]
display(main.status.value_counts().to_frame("copies"))
check("1", "copies outside aggregators", 223_261, len(main), 0, "{:,.0f}")
check("1", "dated copies", 198_579, (main.status == "datada").sum(), 0, "{:,.0f}")
check("1", "share with inherited history (commit before repository)", 0.117,
      (main.first_hash_commit_at < main.repo_created_at).mean(), 0.0005)

In [ ]:
# T0, author and time to first copy by another developer, recomputed from the dated copies
dated = main[main.arrival.notna()].sort_values(["content_hash", "arrival", "repo_full_name", "path"])
origin = dated.drop_duplicates("content_hash")[["content_hash", "arrival", "owner"]].rename(
    columns={"arrival": "t0", "owner": "author"})
other = dated.merge(origin, on="content_hash").query("owner != author")
lag = (other.groupby("content_hash").arrival.min() - other.groupby("content_hash").t0.first()).dt.total_seconds() / 86400
T = pd.read_parquet("data/derived/study_table.parquet")
mine = origin.set_index("content_hash").loc[T.content_hash]
print("skills with a dated copy:", len(origin))
check("1", "T0 identical to the study table", 1.0, (mine.t0.values == pd.to_datetime(T.t0, utc=True).values).mean(), 0)
check("1", "time to first copy identical", 1.0,
      np.isclose(lag.reindex(T.content_hash).fillna(-1).values, T.first_copy_lag_days.fillna(-1).values).mean(), 0)

---
## Part 2 — Window, eligibility, target and split
> 📖 Read [§3.2 and §3.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

The window W is the smallest of 30, 60 and 90 days covering ≥ 80% of the times to first copy, among development
skills observed for ≥ 90 days. A skill is **eligible** if its whole window ends before the snapshot (9 July 2026),
and **copied** if another developer copied it within W days.

In [ ]:
win = S.choose_window(S.load_frame("dev", eligible_only=False))
print(win)
check("2", "window W (days)", 60, win["W"], 0, "{:.0f}")
dev, hold = S.load_frame("dev", w=win["W"]), S.load_frame("holdout", w=win["W"])
check("2", "eligible development skills", 48_299, len(dev), 0, "{:,.0f}")
check("2", "eligible holdout skills", 20_564, len(hold), 0, "{:,.0f}")
check("2", "share copied (holdout)", 0.0775, hold.replicated.mean(), 0.00005, "{:.4f}")
print("authors on both sides:", len(set(dev.owner) & set(hold.owner)), "| max skills per author:", T.groupby("owner").size().max())
show("eda.png")

---
## Part 3 — Exploratory rules
> 📖 Read [§3.5 and §4.1](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

In [ ]:
dev_rules = S.eda_rules(dev.assign(repo_age_days_at_t0=dev.repo_age_days_at_t0.where(dev.repo_age_days_at_t0 >= 0)))
saved = j("phase3/eda_rules.json")
for k in ("constant", "skewed_log1p_nontree", "drop_collinear_nontree"):
    print(k, "identical:", sorted(dev_rules[k]) == sorted(saved[k]), "|", dev_rules[k][:6])
q = pd.qcut(dev.text_lexical_diversity.rank(method="first"), 4, labels=False)
rates = dev.groupby(q).replicated.mean()
check("3", "copied, lowest quartile of lexical diversity", 0.107, rates.iloc[0], 0.0005)
check("3", "copied, highest quartile of lexical diversity", 0.048, rates.iloc[-1], 0.0005)
show("eda_attributes.png"); show("eda_univariate.png", 600)

---
## Part 4 — Spot-check and model selection
> 📖 Read [§3.6 and §4.2](https://github.com/podorize/skillreuse/blob/main/paper/draft/03_methodology.md).

The ⏱ cell re-runs the spot-check with one repetition (the paper used three). Nested tuning is read from the result
files: when all configurations tie, the fastest wins.

In [ ]:
!uv run skillreuse spot --repeats 1 --output-dir results/reproduce/spot | tail -9

In [ ]:
display(pd.read_csv(R / "phase4/spot_summary.csv").round(3))
display(pd.read_csv(R / "phase4/nested_summary.csv").round(3))
display(pd.read_csv(R / "phase4/nested_tests.csv").round(3))
w = j("phase4/winner.json"); print("winner:", w["winner"], "| tied:", w["tied"])
show("model_selection.png")

---
## Part 5 — Single evaluation on the holdout (RQ1) and control
> 📖 Read [§3.7 and §4.3](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md).

The models were frozen with their hashes before the holdout was read. The ⏱ cell re-applies them to the holdout and
checks that the AUCs are identical to the single evaluation.

In [ ]:
!uv run skillreuse reproduce

In [ ]:
h, ctl = j("phase6/holdout_results.json"), j("phase6/control_popularity.json")
print("matches single evaluation:", j("reproduce/holdout_check.json")["matches"])
check("5", "RQ1: text model AUC", 0.645, h["H1"]["auc"])
check("5", "size-only reference AUC", 0.595, h["point"]["size"]["roc_auc"])
check("5", "text minus size", 0.050, h["text_minus_size"]["diff"])
check("5", "origin repository AUC", 0.685, ctl["point"]["origin_pop"]["roc_auc"])
check("5", "origin repository + text AUC", 0.734, ctl["point"]["origin_pop+text"]["roc_auc"])
check("5", "control: gain of the text", 0.049, ctl["C1_popularity"]["diff"])
print("H1 95% CI:", np.round(h["H1"]["ci95"], 3), "| control 95% CI:", np.round(ctl["C1_popularity"]["ci95"], 3))
show("results.png")

---
## Part 6 — Robustness
> 📖 Read [§4.4](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md).

In [ ]:
sens = j("phase7/sensitivity.json")
rows = {k: v for k, v in sens.items() if isinstance(v, dict)}
display(pd.DataFrame(rows).T[["rows", "prevalence", "auc", "ci95"]])
check("6", "temporal split AUC", 0.634, sens["temporal"]["auc"])
check("6", "names unseen AUC", 0.622, sens["name_unseen"]["auc"])

---
## Part 7 — What the model learned (RQ2)
> 📖 Read [§3.8 and §4.5](https://github.com/podorize/skillreuse/blob/main/paper/draft/04_results.md).

In [ ]:
perm = pd.read_csv(R / "phase7/permutation_holdout.csv")
display(perm[perm.feature.str.startswith("family:")].round(4))
check("7", "AUC drop, text-style family", 0.092, perm.set_index("feature").loc["family:Text style", "auc_drop"])
st = pd.read_csv(R / "phase7/family_stability.csv", index_col=0)
print("rank of text style in the 9 development folds:", st.rank(ascending=False).loc["family:Text style"].drop("holdout").tolist())
show("importance.png"); show("shap.png")

---
## Part 8 — What may be claimed
> 📖 Read [§5–6](https://github.com/podorize/skillreuse/blob/main/paper/draft/05_discussion.md).

| Supported | Not supported |
|---|---|
| The text at T0 predicts copying by other developers for unseen authors (AUC 0.645). | That writing style *causes* copying. |
| It says more than text length and more than the origin repository, popularity included. | That the model can flag single skills reliably (precision 10.6%). |
| It holds for future skills (temporal split) and other windows. | Anything about copies deleted before July 2026. |
| Text style is the most important family, in every fold. | The order of the other families. |

In [ ]:
sc = pd.DataFrame(SCORE); sc["ok"] = sc.ok.map({True: "OK", False: "DIFF"})
display(sc)
print(f"{(sc.ok == 'OK').sum()}/{len(sc)} checks within tolerance")